# Task 05 — Reasoning RL with GRPO via Unsloth (Google Colab Free T4)

**Student:** Muhammad Jahid  
**Enrollment No.:** 25TEC2CSML131  
**Task:** Fine-tune a small language model using Group Relative Policy Optimization (GRPO), define a rule-based reward, and inspect reasoning/output changes.

> Run this notebook in Google Colab with **Runtime → Change runtime type → T4 GPU**. The notebook is configured for a small model and a small training subset so it is suitable for experimentation on limited GPU memory. Actual reward values and generated outputs must come from running the notebook; do not report the example/template values as measured results.


## 1. Install dependencies
Use a fresh Colab runtime. If Colab asks to restart after installation, restart and continue from the next cell.


In [ ]:
%%capture
!pip install -U unsloth
!pip install -U trl datasets accelerate bitsandbytes


## 2. Load a compact instruction model
This uses Qwen2.5-0.5B-Instruct with 4-bit loading and LoRA adapters to reduce VRAM use.


In [ ]:
import re, math, os, json
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model loaded. CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 3. Prepare a small math-reasoning dataset
We use GSM8K examples and ask the model to produce a short explanation followed by a final answer in the format `#### number`. A small subset keeps the run manageable. The reward is based on answer correctness and basic output format, not on claiming that hidden chain-of-thought is ground truth.


In [ ]:
raw = load_dataset("openai/gsm8k", "main", split="train[:120]")

SYSTEM_PROMPT = (
    "You are a careful math problem solver. Explain the key calculation briefly, "
    "then give the final numeric answer on a new line in exactly this format: "
    "#### <number>. Do not guess."
)

def extract_gold(answer):
    match = re.search(r"####\s*(-?\d[\d,]*(?:\.\d+)?)", answer)
    return match.group(1).replace(",", "") if match else None

def to_prompt(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "gold_answer": extract_gold(example["answer"]),
    }

dataset = raw.map(to_prompt, remove_columns=raw.column_names)
dataset = dataset.filter(lambda x: x["gold_answer"] is not None)
print("Training examples:", len(dataset))
print(dataset[0]["prompt"])


## 4. Define rule-based reward functions
Correct answers receive the main reward. Outputs with the required final-answer marker receive a smaller format reward. A light brevity penalty discourages excessively long outputs. These are simple task rewards for a demonstration, not a comprehensive measure of reasoning quality.


In [ ]:
def completion_text(completion):
    # TRL may provide completions as strings or chat-message lists.
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict):
                parts.append(str(item.get("content", "")))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(completion)

def answer_number(text):
    matches = re.findall(r"####\s*(-?\d[\d,]*(?:\.\d+)?)", text)
    if not matches:
        # Fallback: take the last standalone number.
        matches = re.findall(r"(?<![\w.])-?\d[\d,]*(?:\.\d+)?", text)
    return matches[-1].replace(",", "") if matches else None

def correctness_reward(prompts, completions, gold_answer, **kwargs):
    rewards = []
    for completion, gold in zip(completions, gold_answer):
        pred = answer_number(completion_text(completion))
        try:
            is_correct = pred is not None and abs(float(pred) - float(gold)) < 1e-6
        except (TypeError, ValueError):
            is_correct = False
        rewards.append(2.0 if is_correct else -0.5)
    return rewards

def format_reward(completions, **kwargs):
    return [0.25 if "####" in completion_text(c) else -0.1 for c in completions]

def concise_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        words = len(completion_text(c).split())
        rewards.append(0.1 if 10 <= words <= 140 else -0.05)
    return rewards

print("Reward functions defined.")


## 5. Configure and run GRPO
GRPO samples multiple completions per prompt and optimizes relative rewards within each group. Training duration is intentionally short for a Colab demonstration. If your runtime runs out of memory, reduce `max_prompt_length`, `max_completion_length`, or `per_device_train_batch_size`.


In [ ]:
training_args = GRPOConfig(
    output_dir="jahid-grpo-output",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=384,
    max_completion_length=192,
    max_steps=30,
    logging_steps=1,
    save_steps=15,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    temperature=0.8,
    beta=0.04,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward, format_reward, concise_reward],
    args=training_args,
    train_dataset=dataset,
)

train_result = trainer.train()
print(train_result)


## 6. Plot the reward curve
The trainer logs metrics during training. The chart below plots the logged reward metric if available. If your installed TRL version uses a different metric name, inspect `trainer.state.log_history` and update the key accordingly. Save the chart for the report.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
display(history.tail(10))

candidate_keys = ["reward", "rewards", "mean_reward", "reward/mean"]
reward_key = next((k for k in candidate_keys if k in history.columns), None)
if reward_key is None:
    print("No standard reward key found. Available metric keys:", list(history.columns))
    print("Use the logged reward metric shown above for your TRL/Unsloth version.")
else:
    curve = history[["step", reward_key]].dropna()
    plt.figure()
    plt.plot(curve["step"], curve[reward_key], marker="o")
    plt.xlabel("Training step")
    plt.ylabel(reward_key)
    plt.title("GRPO reward progression")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("reward_progression.png", dpi=160)
    plt.show()
    print("Saved reward_progression.png")


## 7. Compare a sample before and after training
This cell uses the same prompt before and after training. Since generation is stochastic and training is short, one sample does not prove overall improvement. Compare answer correctness across several held-out questions for a more reliable evaluation.


In [ ]:
sample = raw[0]
sample_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": sample["question"]},
]

def generate_answer(current_model, prompt, max_new_tokens=192):
    text_in = tokenizer.apply_chat_template(
        prompt, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text_in, return_tensors="pt").to(current_model.device)
    with torch.no_grad():
        output = current_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Capture the original/base response before loading the adapter.
# If the model has already been updated in-place by GRPO, reload a fresh
# base model in a separate cell/runtime to obtain a genuine before sample.
print("QUESTION:", sample["question"])
print("REFERENCE ANSWER:", sample["answer"])
print("\nAFTER TRAINING:")
print(generate_answer(model, sample_prompt))


## 8. Save adapter and artifacts
The LoRA adapter can be downloaded from the Colab file browser or saved to Google Drive. Keep the reward plot and a few before/after samples for your final submission.


In [ ]:
model.save_pretrained("jahid-grpo-lora")
tokenizer.save_pretrained("jahid-grpo-lora")

# Download the reward plot if it was generated.
from google.colab import files
import os
if os.path.exists("reward_progression.png"):
    files.download("reward_progression.png")
print("Adapter saved to jahid-grpo-lora/")


## Conclusion template — fill after the run

- **Base model:** Qwen2.5-0.5B-Instruct
- **Method:** GRPO with LoRA adapters
- **Reward design:** answer correctness, required final-answer format, and concise output
- **Training steps:** 30 (unless changed for memory/runtime)
- **Observed reward trend:** Describe the actual plot produced in Section 6; do not assume it rises monotonically.
- **Before/after observation:** Paste actual outputs from Section 7 and state whether the final answer is correct.
- **Limitations:** A small subset and short run are only a proof of concept. Reward hacking, noisy rewards, and a single sample can misrepresent general performance.
